In [ ]:
import pandas as pd
import re

def clean_protocol(val):
    text = str(val).lower()
    match = re.search(r'pcie\s*(\d)', text) # .search() 查找第一个匹配项,\s*表示空格可以有0个或多个,()表示捕获组
    if match:
        return int(match.group(1)) # .group(1) 获取第一个捕获组的内容
    # 尝试找第一个出现的数字
    match = re.search(r'(\d)', text)
    if match:
        return int(match.group(1))
    return 0

def clean_price(val):
    try:
        res = re.sub(r'[^\d.]', '', str(val))
        return float(res) if res else 0.0
    except:
        return 0.0

# 路径
input_path = r'c:\Users\Administrator\Desktop\25三创赛资料\笔记本电脑数据\固态硬盘\固态硬盘汇总-3p.xlsx'
output_path = r'c:\Users\Administrator\Desktop\25三创赛资料\笔记本电脑数据\固态硬盘\固态硬盘汇总-已处理.xlsx'

print(f"正在读取文件: {input_path}")
df = pd.read_excel(input_path)

# 处理列名（防止乱码干扰）
# 我们知道 '协议 (标签)' 通常在倒数第三列或通过名字匹配
target_col = '协议 (标签)'
price_col = '价格'

# 清洗协议
if target_col in df.columns:
    print(f"正在清洗列: {target_col}")
    df['协议_已处理'] = df[target_col].apply(clean_protocol)

# 清洗价格
if price_col in df.columns:
    print(f"正在清洗列: {price_col}")
    df[price_col] = df[price_col].apply(clean_price)


# 保存文件
df.to_excel(output_path, index=False)
print(f"成功保存到: {output_path}")


In [ ]:
import pandas as pd
import re

input_path = r'C:\Users\Administrator\Desktop\25三创赛资料\笔记本电脑数据\固态硬盘\固态硬盘汇总-3p.xlsx'
df = pd.read_excel(input_path)

def clean_protocol(raw):
    text = str(raw).lower()
    match = re.search(r'pcie\s*(\d)', text)
    if match:
        nums = int(match.group(1))
        if nums == 4:
            return 0
        if nums == 5:
            return 1
    return 0

target_col = ''
for col in df.columns:
    if '协议 (标签)' in col:
        target_col = col
        break

if target_col:
    df['Index'] = df[target_col].apply(clean_protocol)

cols = ['Index'] + [c for c in df.columns if c != 'Index']
df = df[cols]

output_path = r'c:\Users\Administrator\Desktop\25三创赛资料\笔记本电脑数据\固态硬盘\固态硬盘汇总-已处理2.xlsx'
df.to_excel(output_path, index=False)
print(f"成功保存到: {output_path}")


        
    

In [ ]:
import pandas as pd
import numpy as np
import re

def execute(dataframe1=None, dataframe2=None, dataframe3=None):
    if dataframe1 is None or dataframe1.empty:
        return pd.DataFrame([{"分析": "无数据", "值": 0}])

    df = dataframe1.copy()
    # 清理列名空格
    df.columns = df.columns.str.strip()

    # 1. 匹配列
    protocol_col = next((c for c in df.columns if "协议" in str(c)), "")
    price_col = next((c for c in df.columns if "价" in str(c)), "")
    cap_col = next((c for c in df.columns if "容量" in str(c) or "标识" in str(c)), "")

    # 2. 提取容量（参照内存脚本逻辑）
    def process_capacity(val):
        try:
            val_str = str(val).upper()
            is_tb = 'TB' in val_str
            nums = re.findall(r'\d+', val_str)
            if not nums: return 0
            num = int(nums[0])
            if is_tb or num <= 8: 
                return num * 1024
            return num
        except:
            return 0

    df['容量_数字'] = df[cap_col].apply(process_capacity)
    df[price_col] = pd.to_numeric(df[price_col], errors='coerce')
    
    # 算出 1GB 多少钱
    df = df[df['容量_数字'] > 0] 
    df['每GB单价'] = df[price_col] / df['容量_数字']

    # 3. 统计结果（完全对照内存脚本的 .agg 格式）
    result = df.groupby(protocol_col).agg({
        '每GB单价': 'mean',
        price_col: 'mean',
        '容量_数字': 'count' 
    }).reset_index().rename(columns={'容量_数字': '样本数'})

    return result.round(2)

# 【修正点】读取原始数据
df_raw = pd.read_excel(r'c:\Users\Administrator\Desktop\25三创赛资料\笔记本电脑数据\固态硬盘\固态硬盘汇总-已处理2.xlsx')

# 【修正点】直接调用函数处理整个 DataFrame，不要用 applymap
# applymap 是把函数应用到每一个格子里，所以报错了
df_result = execute(df_raw)

# 保存统计结果
print("分析结果：")
print(df_result)
df_result.to_excel(r'c:\Users\Administrator\Desktop\25三创赛资料\笔记本电脑数据\固态硬盘\固态硬盘汇总-已处理3.xlsx', index=False)
